# Fine-tune Mistral 7B ให้ decompose ตามเฉลย

## ทำอะไร

สอน Mistral 7B ด้วยเฉลย 111 ชุดที่คนตรวจแล้ว ให้มันซอยข้อความแบบเดียวกับที่คนทำ

## ทำไมถึง fine-tune แทนการปรับ prompt

เกณฑ์คุณภาพในงานวิจัยมี 7 ประเภทข้อผิดพลาด บวก 2 caution — over-segmentation,
context loss, meaning distortion, spurious text, non-atomic split, style drift,
ambiguous pronoun reference, tone change, language level change

การอธิบายทั้งหมดนี้เป็นกฎใน prompt ทำได้จำกัด (prompt ที่ตีพิมพ์มี 12 ข้อแล้วยังมี
over-segmentation 15.9%) แต่เฉลยที่คนแก้แสดง "คำตอบที่ถูก" ครบทุกมิติพร้อมกันในตัวอย่างเดียว
— ทั้งการซอย การคงคำเดิม การแทนสรรพนาม การเก็บบริบท

fine-tune คือการให้โมเดลเรียนจากตัวอย่างเหล่านั้นโดยตรง แทนการอ่านกฎ

## ผลพลอยได้ที่สำคัญกับ VM

พอพฤติกรรมอยู่ใน weights แล้ว prompt ไม่ต้องแบกกฎ 12 ข้อกับตัวอย่างอีก
จาก ~700 tokens เหลือ ~40 บน VM ที่ไม่มี GPU การประมวลผล prompt กินเวลาจริง
ตรงนี้จึงเร็วขึ้นด้วย ไม่ใช่แค่แม่นขึ้น

## baseline ที่ใช้เทียบ

จากเปเปอร์ IAIT (Mistral 7B, prompt `"3-5 points"`, 285 runs ที่ annotate แล้ว)

| | |
|---|---|
| success rate | 65.9% |
| clean (ไม่มี caution) | 46.6% |
| over-segmentation | 15.9% |
| language level change | 21.6% |
| points เฉลี่ย (26 ชุด multi-issue) | 3.38 |

และเฉลยของคนบนทั้ง 111 ชุด: **1.31 points เฉลี่ย** โดย 88 ชุดควรได้ 1 point


## 1. ติดตั้ง

Runtime → Change runtime type → **T4 GPU**


In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate datasets sentence-transformers
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. ข้อมูล

อัปโหลด `data/dataset.csv` (ไฟล์ที่สร้างจาก `scripts/build_dataset.py`)


In [ ]:
import json, re, os, math, random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

from google.colab import files
if not os.path.exists("dataset.csv"):
    files.upload()          # เลือก dataset.csv

df = pd.read_csv("dataset.csv")
df["points"] = df.points_json.map(json.loads)

print(f"{len(df)} responses")
print(f"points เฉลี่ย {df.n_points.mean():.2f}   การกระจาย {df.n_points.value_counts().sort_index().to_dict()}")
print(f"single-issue {(~df.is_multi).sum()} · multi-issue {df.is_multi.sum()}")

## 3. แบ่ง fold

ข้อมูลถูกแบ่งเป็น 5 folds แบบ stratified ไว้แล้ว

รอบแรกให้รัน `HELD_OUT_FOLD = 4` เพื่อดูว่า pipeline ทำงานถูก จากนั้นค่อยวน 0-4
ให้ครบ เพราะ fold เดียวมี multi-issue แค่ 4 ชุด ซึ่งน้อยเกินกว่าจะสรุปได้


In [ ]:
HELD_OUT_FOLD = 4

train_df = df[df.fold != HELD_OUT_FOLD].reset_index(drop=True)
eval_df  = df[df.fold == HELD_OUT_FOLD].reset_index(drop=True)

print(f"train {len(train_df)} · held-out {len(eval_df)}")
print(f"held-out: single-issue {(~eval_df.is_multi).sum()} · multi-issue {eval_df.is_multi.sum()}")

## 4. รูปแบบ prompt

สั้นโดยตั้งใจ กฎทั้งหมดจะถูกเรียนจากตัวอย่าง ไม่ใช่จากการอ่านคำสั่ง

output เป็น JSON array ของ string ตรงกับที่ backend คาดหวัง


In [ ]:
INSTRUCTION = (
    "Split this student feedback into separate issues. "
    "Return only a JSON array of strings."
)


def build_messages(raw_text: str, points: list[str] | None = None):
    """Mistral v0.3 ไม่รองรับ system role จึงรวมคำสั่งไว้ในข้อความของ user"""
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\n{raw_text}"}]
    if points is not None:
        messages.append({
            "role": "assistant",
            "content": json.dumps(points, ensure_ascii=False),
        })
    return messages


example = train_df.iloc[0]
print(build_messages(example.raw_text, example.points)[0]["content"][:400])
print("\n--- target ---")
print(json.dumps(example.points, ensure_ascii=False)[:400])

## 5. โหลดโมเดล

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# ตั้ง HF_TOKEN ที่แถบ Secrets (ไอคอนกุญแจ) — อย่าพิมพ์ลงใน cell
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", token=os.environ["HF_TOKEN"]
)
model.config.use_cache = False
print("โหลดเสร็จ")

## 6. เตรียม dataset

คิด loss เฉพาะส่วนคำตอบ ไม่คิดจาก prompt — ไม่งั้นโมเดลจะเสียกำลังไปกับการเรียน
ทำนายข้อความที่ผู้ใช้พิมพ์ ซึ่งไม่ใช่สิ่งที่ต้องการสอน


In [ ]:
MAX_LEN = 1024   # raw_text ยาวสุด ~2100 ตัวอักษร ≈ 525 tokens บวก output


def to_ids(out):
    """คืน list[int] เสมอ

    apply_chat_template คืนค่าคนละรูปแบบกันในแต่ละเวอร์ชันของ transformers —
    อาจเป็น list[int], list[list[int]], BatchEncoding หรือ tensor
    ตัวนี้ทำให้เหลือรูปแบบเดียว
    """
    if hasattr(out, "input_ids"):
        out = out.input_ids
    elif isinstance(out, dict):
        out = out["input_ids"]
    if hasattr(out, "tolist"):
        out = out.tolist()
    if len(out) and isinstance(out[0], (list, tuple)):
        out = out[0]
    return [int(x) for x in out]


def encode(row):
    prompt_ids = to_ids(tokenizer.apply_chat_template(
        build_messages(row.raw_text), tokenize=True, add_generation_prompt=True))
    full_ids = to_ids(tokenizer.apply_chat_template(
        build_messages(row.raw_text, row.points), tokenize=True))[:MAX_LEN]

    labels = list(full_ids)
    # -100 = ไม่คิด loss ตรงนี้
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100

    return {"input_ids": full_ids, "labels": labels}


train_rows = [encode(r) for r in train_df.itertuples()]
eval_rows = [encode(r) for r in eval_df.itertuples()]

lens = [len(r["input_ids"]) for r in train_rows]
print(f"ความยาว token: เฉลี่ย {np.mean(lens):.0f} · p95 {np.percentile(lens,95):.0f} · สูงสุด {max(lens)}")
truncated = sum(1 for n in lens if n >= MAX_LEN)
print(f"ถูกตัดเพราะยาวเกิน: {truncated} ชุด" + ("  <-- ควรเพิ่ม MAX_LEN" if truncated else ""))

# ตรวจว่า mask ถูก: ต้องมี token ที่คิด loss เหลืออยู่
supervised = sum(1 for x in train_rows[0]["labels"] if x != -100)
print(f"ตัวอย่างแรก: คิด loss {supervised} tokens จากทั้งหมด {len(train_rows[0]['input_ids'])}")
assert supervised > 0

# ตรวจว่าส่วนที่คิด loss คือคำตอบจริง ไม่ใช่ prompt
answer = tokenizer.decode([t for t, l in zip(train_rows[0]["input_ids"], train_rows[0]["labels"]) if l != -100])
print(f"ส่วนที่คิด loss: {answer[:100]!r}")
assert answer.strip().startswith("["), "mask ผิด — ส่วนที่คิด loss ไม่ใช่ JSON array"

In [ ]:
from torch.utils.data import Dataset


class PointsDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]


def collate(batch):
    """pad ให้ยาวเท่ากันในแต่ละ batch — labels pad ด้วย -100 เพื่อไม่ให้คิด loss"""
    longest = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, mask = [], [], []
    for b in batch:
        pad = longest - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [tokenizer.pad_token_id] * pad)
        labels.append(b["labels"] + [-100] * pad)
        mask.append([1] * len(b["input_ids"]) + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(mask),
    }

## 7. LoRA

r เล็กโดยตั้งใจ ข้อมูลมีแค่ 90 ชุด ความจุที่มากเกินไปทำให้ท่องจำแทนที่จะเรียนรู้


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 8. เทรน

ข้อมูล 90 ชุดถือว่าน้อยมาก ความเสี่ยงหลักคือ overfitting จึงประเมินทุก epoch
แล้วเก็บ checkpoint ที่ eval loss ต่ำสุด ไม่ใช่ checkpoint สุดท้าย


In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=f"out_fold{HELD_OUT_FOLD}",
    num_train_epochs=6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,      # กัน overfit: เอา checkpoint ที่ดีที่สุด
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=PointsDataset(train_rows),
    eval_dataset=PointsDataset(eval_rows),
    data_collator=collate,
)

trainer.train()

In [ ]:
hist = pd.DataFrame(trainer.state.log_history)
losses = hist[["epoch", "loss", "eval_loss"]].groupby("epoch").first()
print(losses.to_string())

best = losses.eval_loss.idxmin()
print(f"\neval loss ต่ำสุดที่ epoch {best}")
if best < losses.index.max():
    print("eval loss เริ่มแย่ลงหลังจากนั้น = overfit — load_best_model_at_end จัดการให้แล้ว")

## 9. สร้างผลลัพธ์บน held-out

เทียบ 2 อย่างบนชุดเดียวกันด้วย prompt เดียวกัน — ตัวที่ยังไม่เทรน กับตัวที่เทรนแล้ว
ถ้าไม่เทียบก็บอกไม่ได้ว่าที่ดีขึ้นมาจากการเทรนหรือมาจาก prompt สั้นที่เปลี่ยนไป


In [ ]:
ARRAY_RE = re.compile(r"\[.*?\]", re.DOTALL)


def parse_points(raw: str):
    """คืน (points, status) — ไม่กลบข้อผิดพลาด

    ลอง parse ตรง ๆ ก่อน ถ้าไม่ได้ค่อยดึง JSON array ตัวแรกที่เจอในข้อความ
    """
    text = raw.strip()
    candidates = [(text, "ok")]
    match = ARRAY_RE.search(text)
    if match:
        candidates.append((match.group(0), "ok_fallback"))

    for candidate, status in candidates:
        try:
            obj = json.loads(candidate)
        except Exception:
            continue
        if isinstance(obj, list) and all(isinstance(x, str) for x in obj):
            points = [p.strip() for p in obj if p.strip()]
            if points:
                return points, status

    return [], "parse_error"


@torch.inference_mode()
def generate(raw_text: str, use_adapter: bool) -> tuple[list[str], str]:
    if use_adapter:
        out = _gen(raw_text)
    else:
        # ปิด adapter ชั่วคราวเพื่อให้ได้ base model ด้วย prompt เดียวกัน
        with model.disable_adapter():
            out = _gen(raw_text)
    return parse_points(out)


def _gen(raw_text: str) -> str:
    ids = torch.tensor([to_ids(tokenizer.apply_chat_template(
        build_messages(raw_text), tokenize=True, add_generation_prompt=True))]).to(model.device)
    out = model.generate(
        ids, max_new_tokens=512, do_sample=False,
        repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


model.eval(); model.config.use_cache = True

results = []
for r in eval_df.itertuples():
    for tag, use_adapter in (("base", False), ("finetuned", True)):
        pts, status = generate(r.raw_text, use_adapter)
        results.append({
            "feedback_id": r.feedback_id, "model": tag,
            "raw_text": r.raw_text, "gold": r.points, "pred": pts,
            "status": status, "gold_n": len(r.points), "pred_n": len(pts),
            "is_multi": r.is_multi,
        })
    print(f"  id={r.feedback_id} gold={len(r.points)} "
          f"base={results[-2]['pred_n']} ft={results[-1]['pred_n']}")

res = pd.DataFrame(results)

## 10. วัดผลตามเกณฑ์ในงานวิจัย

proxy อัตโนมัติที่จับคู่กับ error category ในเปเปอร์ได้ตรง ๆ

| Metric | จับ error ประเภท |
|---|---|
| `count_exact` / `over_split` | Over-segmentation, Non-atomic split |
| `coverage` | Context loss |
| `fidelity` | Style drift, Language level change |
| `pronoun_rate` | Ambiguous pronoun reference |
| `parse_error` | Spurious text |

Meaning distortion กับ Non-atomic split ต้องใช้คนตรวจ วัดอัตโนมัติไม่ได้

### ค่าเป้าหมาย (วัดจากการป้อนเฉลยเข้าไปเป็นคำตอบ)

| metric | เพดาน |
|---|---|
| `count_exact` | 1.000 |
| `over_split` | 0.000 |
| `coverage` | 0.934 |
| `fidelity` | 0.832 |
| `pronoun_rate` | 0.033 |

นี่คือคะแนนที่ได้ถ้าตอบตรงกับเฉลยเป๊ะ ไม่ใช่ 1.0 เพราะ coverage กับ fidelity
เทียบกับข้อความดิบ และเฉลยมีการแทนสรรพนามกับตัดคำซ้ำออกบ้าง

`pronoun_rate` ของข้อความดิบคือ 0.043 ส่วนเฉลยคือ 0.033 — เฉลยแทนสรรพนามด้วยคำนามแล้ว
ถ้าโมเดลได้ค่าใกล้ 0.043 แปลว่ามันคัดลอกมาเฉย ๆ ไม่ได้แก้สรรพนามตามที่ต้องการ


In [ ]:
WORD = re.compile(r"[a-z0-9']+")
STOP = set("""a an the and or but if then than that this these those is are was were be been being am
do does did doing have has had having i me my we our you your he she it they them his her its their to
of in on at for with as by from about into over after before between out up down off not no so very
can will just should now more most some such only own same too s t don""".split())
PRONOUNS = {"it", "this", "that", "they", "them", "she", "he", "her", "his", "these", "those"}

words = lambda t: WORD.findall(t.lower())
content = lambda t: {w for w in words(t) if w not in STOP and len(w) > 2}
bigrams = lambda t: set(zip(words(t), words(t)[1:]))


def score(row):
    pred, gold, src = row.pred, row.gold, row.raw_text
    if not pred:
        return pd.Series({"coverage": np.nan, "fidelity": np.nan,
                          "pronoun_rate": np.nan, "count_exact": False,
                          "over_split": np.nan})

    joined = " ".join(pred)
    src_c = content(src)
    out_bg = bigrams(joined)

    return pd.Series({
        # เก็บเนื้อหาไว้ครบไหม
        "coverage": len(src_c & content(joined)) / len(src_c) if src_c else np.nan,
        # คัดลอกคำเดิมหรือเขียนใหม่
        "fidelity": len(out_bg & bigrams(src)) / len(out_bg) if out_bg else np.nan,
        # สรรพนามที่ไม่ได้แทนด้วยคำนาม
        "pronoun_rate": sum(1 for w in words(joined) if w in PRONOUNS) / max(len(words(joined)), 1),
        "count_exact": len(pred) == len(gold),
        "over_split": len(pred) > len(gold),
    })


res = pd.concat([res, res.apply(score, axis=1)], axis=1)

summary = res.groupby("model").agg(
    n=("feedback_id", "count"),
    pred_points=("pred_n", "mean"),
    count_exact=("count_exact", "mean"),
    over_split=("over_split", "mean"),
    coverage=("coverage", "mean"),
    fidelity=("fidelity", "mean"),
    pronoun_rate=("pronoun_rate", "mean"),
    parse_error=("status", lambda s: (s == "parse_error").mean()),
).round(3)

print(f"เฉลยของ held-out: {res.groupby('model').gold_n.mean().iloc[0]:.2f} points")
print("เพดาน (ป้อนเฉลยเป็นคำตอบ): count_exact 1.000 · over_split 0.000 · "
      "coverage 0.934 · fidelity 0.832 · pronoun_rate 0.033\n")
print(summary.to_string())

# ตัวชี้ขาด: ข้อความที่ควรได้ 1 point มันซอยเกินกี่ %
single = res[~res.is_multi]
print(f"\nข้อความประเด็นเดียว (n={single.feedback_id.nunique()}) — สัดส่วนที่ซอยเกิน")
print(single.groupby("model").over_split.mean().round(3).to_string())

multi = res[res.is_multi]
print(f"\nข้อความหลายประเด็น (n={multi.feedback_id.nunique()}) — สัดส่วนที่นับจุดถูก")
print(multi.groupby("model").count_exact.mean().round(3).to_string())
print("  n น้อยมาก ต้องรันครบ 5 folds ก่อนสรุป")

## 11. ดูผลจริงเทียบกัน

In [ ]:
for fid in eval_df.feedback_id.head(4):
    rows = res[res.feedback_id == fid]
    print("=" * 78)
    print("RAW :", rows.iloc[0].raw_text[:200])
    print(f"\nเฉลย ({rows.iloc[0].gold_n} จุด):")
    for p in rows.iloc[0].gold: print("   *", p[:100])
    for _, r in rows.iterrows():
        print(f"\n{r.model} ({r.pred_n} จุด, {r.status}):")
        for p in r.pred: print("   -", p[:100])
    print()

## 12. บันทึก adapter

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

save_dir = f"/content/drive/MyDrive/Capstone/adapters/fold{HELD_OUT_FOLD}"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

res.drop(columns=["raw_text"]).to_csv(
    f"/content/drive/MyDrive/Capstone/finetune_results_fold{HELD_OUT_FOLD}.csv", index=False
)
print("บันทึกที่", save_dir)
print(f"ขนาด adapter: {sum(f.stat().st_size for f in __import__('pathlib').Path(save_dir).glob('*')) / 1e6:.1f} MB")

## ถัดไป

1. รันซ้ำด้วย `HELD_OUT_FOLD = 0,1,2,3` แล้วรวมผล — fold เดียวมี multi-issue แค่ 4 ชุด
   สรุปไม่ได้
2. ถ้าผลดีขึ้นจริง ให้ merge adapter แล้วแปลงเป็น GGUF เพื่อรันบน VM
3. เอาตัวอย่างที่ผิดไปให้คนอ่าน เพื่อดู error ประเภทที่วัดอัตโนมัติไม่ได้
   (meaning distortion, non-atomic split)
